## <center>CITS5508 Assignment 2</center>

Author: Seng Qi Ming (25303739)

 - [Ensemble Learning for Heart Disease Prediction](#part-1-ensemble-learning-for-heart-disease-prediction)
 - [Clustering of Faces](#part-2-clustering-of-faces)

In this assignment we will first use ensemble learning for heart disease prediction, for the next part we will explore on technique for clustering of faces.

### Importing necessary libraries

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, VotingClassifier
from sklearn.metrics import classification_report

## Part 1: Ensemble Learning for Heart Disease Prediction

In this task, we will compare a range of ensemble methods for predicting the presence of heart disease
in patients

### 1.1: Load the dataset & Inspect the data
We will first load the dataset using the `processed.cleveland.data`

In [2]:
column_names = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num']

data = pd.read_csv('processed.cleveland.data', names=column_names, na_values='?')

In [3]:
data.describe()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
count,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,299.000000,301.000000,303.000000
mean,54.438944,0.679868,3.158416,131.689769,246.693069,0.148515,0.990099,149.607261,0.326733,1.039604,1.600660,0.672241,4.734219,0.937294
std,9.038662,0.467299,0.960126,17.599748,51.776918,0.356198,0.994971,22.875003,0.469794,1.161075,0.616226,0.937438,1.939706,1.228536
min,29.000000,0.000000,1.000000,94.000000,126.000000,0.000000,0.000000,71.000000,0.000000,0.000000,1.000000,0.000000,3.000000,0.000000
25%,48.000000,0.000000,3.000000,120.000000,211.000000,0.000000,0.000000,133.500000,0.000000,0.000000,1.000000,0.000000,3.000000,0.000000
50%,56.000000,1.000000,3.000000,130.000000,241.000000,0.000000,1.000000,153.000000,0.000000,0.800000,2.000000,0.000000,3.000000,0.000000
75%,61.000000,1.000000,4.000000,140.000000,275.000000,0.000000,2.000000,166.000000,1.000000,1.600000,2.000000,1.000000,7.000000,2.000000
max,77.000000,1.000000,4.000000,200.000000,564.000000,1.000000,2.000000,202.000000,1.000000,6.200000,3.000000,3.000000,7.000000,4.000000


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    float64
 1   sex       303 non-null    float64
 2   cp        303 non-null    float64
 3   trestbps  303 non-null    float64
 4   chol      303 non-null    float64
 5   fbs       303 non-null    float64
 6   restecg   303 non-null    float64
 7   thalach   303 non-null    float64
 8   exang     303 non-null    float64
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    float64
 11  ca        299 non-null    float64
 12  thal      301 non-null    float64
 13  num       303 non-null    int64  
dtypes: float64(13), int64(1)
memory usage: 33.3 KB


In [5]:
n_examples = data.shape[0]
n_features = data.shape[1]

numeric_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']
categorical_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']

print(f'Number of examples: {n_examples}')
print(f'Number of features (including target): {n_features}')
print(f'Numeric features: {numeric_features}')
print(f'Categorical features: {categorical_features}')

data['num_binary'] = (data['num'] > 0).astype(int)
print('Target distribution (original):')
print(data['num'].value_counts().sort_index())
print('Target distribution (binary):')
print(data['num_binary'].value_counts().sort_index())

Number of examples: 303
Number of features (including target): 14
Numeric features: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']
Categorical features: ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']
Target distribution (original):
num
0    164
1     55
2     36
3     35
4     13
Name: count, dtype: int64
Target distribution (binary):
num_binary
0    164
1    139
Name: count, dtype: int64


### Observation
- There are 303 examples (which mean 303 patients).
- There are 13 features (excluding the target). 
    - Numeric: `age`, `trestbps`, `chol`, `thalach`, `oldpeak`, `ca`.
    - Categorical: `sex`, `cp`, `fbs`, `restecg`, `exang`, `slope`, `thal`.

#### 1.2 Splitting of dataset
Split the dataset into a training and testing set. Use 80% of the data for training, and 20% for
testing. Make sure to stratify your split. 
For row with missing values, we will drop the row

In [6]:
data = data.dropna()

In [7]:
# Split to X and Y first 
X = data.drop(columns=['num', 'num_binary'])
y = data['num_binary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Train samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

Train samples: 237, Test samples: 60


#### 1.3 Using a single Decision Tree classifier on our dataset

Fit a single Decision Tree classifier using scikit-learn as a baseline and evaluate its performance on the test set.

In [8]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

y_pred = dt.predict(X_test)

print(f'\nClasification Report \n {classification_report(y_true=y_test, y_pred=y_pred)}')




Clasification Report 
               precision    recall  f1-score   support

           0       0.71      0.75      0.73        32
           1       0.69      0.64      0.67        28

    accuracy                           0.70        60
   macro avg       0.70      0.70      0.70        60
weighted avg       0.70      0.70      0.70        60



#### 1.4 Fitting of Random Forest Classifier and AdaBoost Classifier

In [9]:
rf = RandomForestClassifier(n_estimators=500, random_state=42)
rf.fit(X_train, y_train)

adaBoost = AdaBoostClassifier(n_estimators=500, random_state=42)
adaBoost.fit(X_train, y_train)

AdaBoostClassifier(n_estimators=500, random_state=42)

#### 1.5 Combining both Random Forest  classifier and Adaboost Classifier using Voting Classifier

In [ ]:
vc = VotingClassifier(estimators=[('rf', rf), ('adaBoost', adaBoost)])

## Part 2: Clustering of Faces

